In [1]:
x = 1
y = 25
print(x+y)

26


In [3]:
import sys
print(sys.executable)

c:\Program Files\Python311\python.exe


In [ ]:
!pip install pdfplumber pandas openpyxl ipywidgets

In [1]:
# ==========================================
# INSTALL THESE FIRST (RUN ONCE)
# ==========================================
# pip install pdfplumber pandas openpyxl

import tkinter as tk
from tkinter import ttk, filedialog, messagebox
import pdfplumber
import pandas as pd
from pathlib import Path
import re


# ==========================================
# MAIN WINDOW
# ==========================================
root = tk.Tk()
root.title("PDF Table Extractor")
root.geometry("800x250")


# ==========================================
# VARIABLES
# ==========================================
selected_pdf_path = tk.StringVar()


# ==========================================
# FILE SELECTOR
# ==========================================
def select_pdf_file():
    file_path = filedialog.askopenfilename(
        title="Select PDF File",
        filetypes=[("PDF Files", "*.pdf")]
    )

    if file_path:
        selected_pdf_path.set(file_path)


# ==========================================
# HELPER FUNCTION
# ==========================================
def line_text(words):
    return " ".join(
        w["text"] for w in sorted(words, key=lambda x: x["x0"])
    ).strip()


# ==========================================
# PDF EXTRACTION
# ==========================================
def extract_table_by_positions(pdf_path):

    rows = []

    # Column boundaries
    columns = {
        "Title": (0, 275),
        "Case #": (275, 375),
        "CA #": (375, 490),
        "Action Type": (490, 640),
        "Result": (640, 900),
    }

    with pdfplumber.open(pdf_path) as pdf:

        for page in pdf.pages:

            words = page.extract_words(
                x_tolerance=2,
                y_tolerance=3
            )

            # Find table header
            header_tops = [
                w["top"]
                for w in words
                if w["text"].lower() == "title"
            ]

            if not header_tops:
                continue

            table_start = min(header_tops) + 15

            # Only words below header
            table_words = [
                w for w in words
                if w["top"] > table_start
            ]

            # Group by line
            lines = {}

            for w in table_words:
                y = round(w["top"] / 3) * 3
                lines.setdefault(y, []).append(w)

            # Process lines
            for y in sorted(lines):

                line_words = lines[y]

                full_line = line_text(line_words)

                # Skip junk/header lines
                if full_line.isdigit():
                    continue

                if "Supreme Court of California" in full_line:
                    continue

                if "Results from the petition conference" in full_line:
                    continue

                row = {
                    col: ""
                    for col in columns
                }

                # Assign words into columns
                for col, (x_min, x_max) in columns.items():

                    col_words = [
                        w for w in line_words
                        if x_min <= w["x0"] < x_max
                    ]

                    row[col] = line_text(col_words)

                # Detect new row
                is_new_case = bool(
                    re.search(r"S\d{6}", row["Case #"])
                )

                if is_new_case:

                    rows.append(row)

                elif rows:
                    # Continuation line
                    for col in columns:

                        if row[col]:

                            rows[-1][col] = (
                                rows[-1][col]
                                + " "
                                + row[col]
                            ).strip()

    return pd.DataFrame(rows)


# ==========================================
# SAVE TO EXCEL
# ==========================================
def extract_and_save():

    pdf_path = selected_pdf_path.get()

    if not pdf_path:
        messagebox.showerror(
            "Error",
            "Please select a PDF file first."
        )
        return

    try:

        df = extract_table_by_positions(pdf_path)

        if df.empty:
            messagebox.showwarning(
                "No Data",
                "No table rows were found."
            )
            return

        # Ask user where to save the Excel file
        output_file = filedialog.asksaveasfilename(
            title="Save Excel File",
            defaultextension=".xlsx",
            filetypes=[("Excel Files", "*.xlsx")],
            initialfile=Path(pdf_path).stem + ".xlsx"
        )

        # User cancelled save dialog
        if not output_file:
            return

        df.to_excel(output_file, index=False)

        messagebox.showinfo(
            "Success",
            f"Excel file saved:\n\n{output_file}"
        )

    except Exception as e:

        messagebox.showerror(
            "Error",
            str(e)
        )


# ==========================================
# GUI LAYOUT
# ==========================================
title_label = ttk.Label(
    root,
    text="PDF Table Extractor",
    font=("Arial", 16, "bold")
)

title_label.pack(pady=15)


path_entry = ttk.Entry(
    root,
    textvariable=selected_pdf_path,
    width=90
)

path_entry.pack(padx=20, pady=10)


browse_button = ttk.Button(
    root,
    text="Browse PDF",
    command=select_pdf_file
)

browse_button.pack(pady=5)


extract_button = ttk.Button(
    root,
    text="Extract Table and Save to Excel",
    command=extract_and_save
)

extract_button.pack(pady=20)


# ==========================================
# START GUI
# ==========================================
root.mainloop()